# 02 · Fair share, job priority and preemption (Slurm-style)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/cluster-scheduling/02_fair_share_priority.ipynb) &nbsp; 📖 Concept deck: [Slurm for GPU clusters](https://nilesh-salpe.github.io/visual-ai/cluster-scheduling/02-slurm/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** decayed usage with a half-life, the classic fair-share factor F = 2^(−U/S), the multifactor priority as a weighted sum, and choosing which running jobs to preempt.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU in plain Python, so no GPU or cluster is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import math
HALF_LIFE_DAYS = 7          # Slurm's default PriorityDecayHalfLife is 7 days

## Task 1: Usage with a half-life

Each usage record is `(day, gpu_hours)`. Usage decays so that it counts half after one half-life: return `Σ gpu_hours · 0.5^((now − day) / half_life)`.

In [ ]:
def decayed_usage(records, now, half_life=HALF_LIFE_DAYS):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_decayed_usage(f):
    assert abs(f([(0, 100)], 7) - 50) < 1e-9, "one half-life halves the usage"
    assert abs(f([(0, 100), (14, 40)], 14) - 65) < 1e-9
    assert f([], 3) == 0

check("decayed_usage", _test_decayed_usage, decayed_usage)

## Task 2: Classic fair-share factor

Given each user's decayed `usage` and allocated `shares` (dicts), normalise both to fractions of their totals (U and S) and return `{user: 2 ** (-U / S)}`. A user who used exactly their share gets 0.5; under-served users approach 1, heavy users approach 0. (Slurm's default since 19.05 is the rank-based Fair Tree algorithm; classic is still available.)

In [ ]:
def classic_fair_share(usage, shares):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_classic_fair_share(f):
    F = f({"ana": 50, "bo": 50}, {"ana": 1, "bo": 1})
    assert abs(F["ana"] - 0.5) < 1e-9 and abs(F["bo"] - 0.5) < 1e-9, "used exactly their share → 0.5"
    F = f({"ana": 90, "bo": 10}, {"ana": 1, "bo": 1})
    assert F["bo"] > 0.5 > F["ana"], "the lighter user gets the higher factor"
    assert abs(F["ana"] - 2 ** (-0.9 / 0.5)) < 1e-9
    F = f({"ana": 0, "bo": 10}, {"ana": 1, "bo": 3})
    assert F["ana"] == 1.0

check("classic_fair_share", _test_classic_fair_share, classic_fair_share)

## Task 3: Multifactor priority

Slurm's multifactor priority is a weighted sum of factors that each lie in [0, 1], minus the job's nice value. `factors` and `weights` are dicts keyed by factor name (age, fairshare, jobsize, partition, qos); a missing weight means 0 (Slurm's default weight). Return the integer `round(Σ weight · factor) − nice`. Then sort a list of jobs by priority, highest first.

In [ ]:
def job_priority(factors, weights, nice=0):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_job_priority(f):
    w = {"age": 1000, "fairshare": 10000, "qos": 5000}
    assert f({"age": 0.5, "fairshare": 0.8, "qos": 1.0}, w) == 13500
    assert f({"age": 0.5, "fairshare": 0.8, "qos": 1.0, "jobsize": 1.0}, w) == 13500, "a factor with no weight contributes nothing"
    assert f({"fairshare": 0.25}, w, nice=100) == 2400

check("job_priority", _test_job_priority, job_priority)

## Task 4: Whom to preempt

A high-priority job needs `need` GPUs and only `free` are idle. `running` is a list of `(job_id, gpus, priority)` for preemptable jobs. Preempt the **lowest-priority** jobs first (ties: the smaller job first) until enough GPUs are free; return their ids in that order. If even preempting everything would not free enough, return `[]` and preempt nothing.

In [ ]:
def pick_preemptees(running, need, free):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 4: run after implementing
def _test_pick_preemptees(f):
    running = [("a", 8, 100), ("b", 2, 10), ("c", 4, 10), ("d", 8, 50)]
    assert f(running, 8, 2) == ["b", "c"], "lowest priority first, smaller first on ties, stop once 8 GPUs are free"
    assert f(running, 4, 4) == []
    assert f(running, 40, 0) == [], "cannot free 40 GPUs: preempt nothing"

check("pick_preemptees", _test_pick_preemptees, pick_preemptees)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Usage with a half-life
def decayed_usage_solution(records, now, half_life=HALF_LIFE_DAYS):
    return sum(h * 0.5 ** ((now - d) / half_life) for d, h in records)

check("decayed_usage (reference)", _test_decayed_usage, decayed_usage_solution)

In [ ]:
# @title Solution for Task 2: Classic fair-share factor
def classic_fair_share_solution(usage, shares):
    tu, ts = sum(usage.values()) or 1, sum(shares.values())
    return {u: 2 ** (-(usage[u] / tu) / (shares[u] / ts)) for u in shares}

check("classic_fair_share (reference)", _test_classic_fair_share, classic_fair_share_solution)

In [ ]:
# @title Solution for Task 3: Multifactor priority
def job_priority_solution(factors, weights, nice=0):
    return round(sum(weights.get(k, 0) * v for k, v in factors.items())) - nice

check("job_priority (reference)", _test_job_priority, job_priority_solution)

In [ ]:
# @title Solution for Task 4: Whom to preempt
def pick_preemptees_solution(running, need, free):
    out = []
    for jid, g, _ in sorted(running, key=lambda r: (r[2], r[1])):
        if free >= need:
            break
        out.append(jid); free += g
    return out if free >= need else []

check("pick_preemptees (reference)", _test_pick_preemptees, pick_preemptees_solution)

---
## Experiments
How a heavy week fades: the fair-share factor of a user who ran 1,000 GPU-hours on day 0, against a colleague with steady light use, over three weeks.

In [ ]:
for day in [0, 3, 7, 14, 21]:
    heavy = decayed_usage_solution([(0, 1000)], day)
    light = decayed_usage_solution([(d, 20) for d in range(day + 1)], day)
    F = classic_fair_share_solution({"heavy": heavy, "light": light}, {"heavy": 1, "light": 1})
    print(f"day {day:2}: usage heavy {heavy:6.0f} light {light:5.0f}   F heavy {F['heavy']:.2f} light {F['light']:.2f}")

---
## Takeaways
* Fair share compares **recent** usage (decayed with a half-life, 7 days by default) with allocated shares.
* Priority is a weighted sum of normalised factors; weights default to 0, so the configuration decides what matters.
* Preemption picks victims by priority; with `PreemptMode=REQUEUE` they go back to the queue, so checkpointing decides how much work is lost.